# 03. KOSPI 전략 백테스트 + 리스크 측정 보고서

**1부 — 전략:** KOSPI 시가총액 상위 100 종목(해마다 다시 선정)에서 02 노트북의 추세추종 전략을 돌리고 성과 비교
**2부 — 리스크:** 증권사 리스크관리팀이 매일 보는 형태로, 선택한 전략의 위험을 측정·검증·보고

| 2부 구성 | 리스크관리팀이 묻는 질문 |
|---|---|
| ① VaR · ES 시계열 | 평소 하루 최대 얼마나 잃을 수 있나? 최악의 날들은 평균 얼마나 나쁜가? |
| ② VaR 사후검증 | 우리 VaR 모델은 믿을 만한가? (Kupiec 검정, 바젤 신호등) |
| ③ 현재 포트폴리오 VaR | 지금 들고 있는 종목 기준으로 3가지 방법(역사적·분산공분산·몬테카를로) 비교 |
| ④ 리스크 기여도 | 위험이 어느 종목에서 나오나? 분산효과는 얼마나 되나? |
| ⑤ 스트레스 테스트 | 코로나·계엄 같은 사건이 지금 다시 오면? KOSPI −20%면? |
| ⑥ 유동성 리스크 | 운용금액이 100억이라면 다 파는 데 며칠 걸리나? |
| ⑦ 한도 대시보드 | 사전에 정한 한도 안에서 운용되고 있나? |

엔진·리스크 계산 코드는 `src/backtest.py`, `src/risk.py` 에 있음 (함수마다 설명 주석).


## 0. 설정

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pykrx import stock

sys.path.insert(0, "..")
from src import backtest as bt, risk as rk

START, END, WARMUP = "20160101", "20251231", "20140101"
TOP_N = 100                 # 해마다 1월 KOSPI 시총 상위 100
MAX_POS = 10                # 최대 보유 종목 수
STRATS = ["L2", "M1", "M2", "M7"]   # 비교할 전략 (02 결과 상위권)
RISK_STRAT = "M7"           # 2부 리스크 보고서 대상 전략
REPORT_DATE = None          # None → 보유 종목이 있는 가장 최근 날짜 자동 선택

P = dict(bt.PARAMS)         # 전략 파라미터 (바꿔보려면 P["ATR_MULT"] = 3 처럼)

# 리스크 설정
VAR_WINDOW = 250            # VaR 추정에 쓰는 과거 거래일 수 (약 1년)
AUM = 100e8                 # 가정 운용금액 100억 (유동성 리스크 계산용)
PARTICIPATION = 0.10        # 하루 거래대금의 10%까지만 매도 가능하다고 가정
LIMITS = dict(VAR99=0.025,  # 1일 VaR99 ≤ 2.5%
              MDD=-0.20,    # 최대 낙폭 ≥ −20%
              SINGLE=0.20,  # 종목당 비중 ≤ 20%
              LIQ_DAYS=3)   # 청산 소요일 ≤ 3일

SCENARIOS = {               # 역사적 스트레스 시나리오 (기간은 KOSPI 고점→저점 근사)
    "2018 10월 급락":       ("2018-10-01", "2018-10-29"),
    "코로나 폭락":          ("2020-02-17", "2020-03-19"),
    "2022 긴축 하락장":     ("2022-01-03", "2022-09-30"),
    "2024-08 블랙먼데이":   ("2024-08-01", "2024-08-05"),
    "비상계엄":             ("2024-12-03", "2024-12-09"),
    "미국 상호관세":        ("2025-04-02", "2025-04-09"),
}
os.makedirs("data", exist_ok=True); os.makedirs("results", exist_ok=True)

## 1. 데이터 (처음 30~40분, 이후 캐시 — 02 노트북과 캐시 공유)

In [ ]:
def load_cached(name, fn):
    path = f"data/{name}.pkl"
    if os.path.exists(path):
        return pd.read_pickle(path)
    df = fn(); df.to_pickle(path); time.sleep(0.5)
    return df

years = range(int(START[:4]), int(END[:4]) + 1)
yearly_uni = {}
for y in years:
    d = stock.get_nearest_business_day_in_a_week(f"{y}0102", prev=False)
    cap = load_cached(f"cap_{d}", lambda: stock.get_market_cap(d, market="KOSPI"))
    yearly_uni[y] = list(cap["시가총액"].sort_values(ascending=False).index[:TOP_N])

tickers = sorted({t for u in yearly_uni.values() for t in u})
print("일봉 받을 종목 수:", len(tickers))
ohlcv = {}
for i, t in enumerate(tickers):
    ohlcv[t] = load_cached(f"ohlcv_{t}", lambda: stock.get_market_ohlcv(WARMUP, END, t))
    if i % 20 == 0: print(f"{i}/{len(tickers)}")

kospi = stock.get_index_ohlcv(WARMUP, END, "1001")["종가"]
events = pd.read_csv("../data_events/events.csv", parse_dates=["date"])
names = {}
def name(t):
    if t not in names:
        try: names[t] = stock.get_market_ticker_name(t)
        except Exception: names[t] = t
    return names[t]

## 2. 지표 계산

In [ ]:
ind = {t: bt.indicators(df, P) for t, df in ohlcv.items() if len(df) > 300}
days = kospi.loc[START:END].index
dec_days = bt.make_decision_days(days, START, END)
km = kospi.loc[START:END]
print("지표 종목:", len(ind), "/ 판단일:", len(dec_days))

# 1부. 전략 백테스트 — KOSPI 상위 100

In [ ]:
curves, trades, holds, rows = {}, {}, {}, {}
curves["L0 Hold"] = bt.run_hold(ind, yearly_uni, days, P)
rows["L0 Hold"] = bt.perf_metrics(curves["L0 Hold"], km)
for s in STRATS:
    eq, tr, hd = bt.run(ind, yearly_uni, days, dec_days, s, MAX_POS, P, events)
    curves[s], trades[s], holds[s] = eq, tr, hd
    rows[s] = bt.perf_metrics(eq, km, tr)
    tr.to_csv(f"results/03_trades_{s}.csv", index=False)
curves["KOSPI"] = km / km.iloc[0]
rows["KOSPI"] = bt.perf_metrics(curves["KOSPI"], km)
perf = pd.DataFrame(rows).T
bt.fmt_perf(perf)

In [ ]:
ax = pd.DataFrame(curves).plot(figsize=(12, 5), logy=True, title="KOSPI Top100 - cumulative (log)")
for r in events.itertuples():
    if r.category in ("election", "politics"):
        ax.axvline(r.date, color="red", ls=":", alpha=.35, lw=1)
ax.grid(alpha=.3); plt.show()

# 2부. 리스크 측정 보고서

대상 전략의 **일별 수익률**과 **보유 종목**으로 리스크를 측정해요.
VaR·ES 는 모두 **손실을 양수**로 표시 (예: VaR99 = 2.1% → 하루에 2.1% 넘게 잃을 확률이 1%).

In [ ]:
eq = curves[RISK_STRAT]
r = eq.pct_change().dropna()
hd = holds[RISK_STRAT]
stock_cols = [c for c in hd.columns if c != "cash"]
print(f"대상 전략: {RISK_STRAT} / 기간 {r.index[0].date()} ~ {r.index[-1].date()} / 평균 주식 비중 {(1 - hd['cash'] / hd.sum(axis=1)).mean():.0%}")

## ① VaR · ES 시계열

- **역사적 시뮬레이션(HS):** 과거 250일 실제 수익률을 줄 세워 하위 1% 지점 → 분포 가정 없음, 두꺼운 꼬리 반영
- **분산-공분산(Param):** 정규분포 가정, z × σ → 계산이 빠르지만 폭락 위험을 과소평가하기 쉬움
- **ES(기대손실):** VaR 를 넘는 최악의 날들의 평균 → 바젤 FRTB 가 VaR 대신 채택한 지표 (97.5%)

In [ ]:
vr = pd.DataFrame({
    "HS VaR99": rk.var_hist(r, VAR_WINDOW, 0.99), "Param VaR99": rk.var_param(r, VAR_WINDOW, 0.99),
    "HS VaR95": rk.var_hist(r, VAR_WINDOW, 0.95), "HS ES97.5": rk.es_hist(r, VAR_WINDOW, 0.975),
    "Param ES97.5": rk.es_param(r, VAR_WINDOW, 0.975)})
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(r.index, -r.clip(upper=0), color="lightgray", width=1.5, label="daily loss")
vr[["HS VaR99", "Param VaR99", "HS ES97.5"]].plot(ax=ax, lw=1.2)
exc = rk.exceptions(r, vr["HS VaR99"])
ax.scatter(exc[exc].index, -r.loc[exc[exc].index], color="red", s=12, zorder=5, label="HS VaR99 exception")
ax.set_title(f"{RISK_STRAT}: 1-day VaR / ES vs actual losses"); ax.legend(); ax.grid(alpha=.3); plt.show()

latest = vr.dropna().iloc[-1]
print("가장 최근 값:"); print(latest.map("{:.2%}".format).to_string())

## ② VaR 사후검증 (백테스팅)

VaR99 라면 100일 중 1일 정도만 VaR 를 넘는 손실(예외)이 나와야 정상.
- **Kupiec 검정:** 실제 예외 비율이 1% 와 통계적으로 다른가? p-value < 0.05 → 모델 기각
- **바젤 신호등:** 최근 250일 예외 0~4개 녹색 / 5~9개 황색(자본 가산) / 10개↑ 적색(모델 교체)

In [ ]:
bt99 = rk.backtest_table(r, {"HS VaR99": vr["HS VaR99"], "Param VaR99": vr["Param VaR99"]}, 0.99)
bt95 = rk.backtest_table(r, {"HS VaR95": vr["HS VaR95"], "Param VaR95": rk.var_param(r, VAR_WINDOW, 0.95)}, 0.95)
bt_all = pd.concat([bt99, bt95])
bt_all.style.format({"예상 예외 수": "{:.1f}", "예외율": "{:.2%}", "Kupiec LR": "{:.2f}", "p-value": "{:.3f}"})

## ③ 현재 포트폴리오 VaR — 세 가지 방법 비교

보고 기준일에 **실제로 들고 있는 종목·비중**으로 계산 (현금은 위험 0).
몬테카를로는 종목 간 상관관계를 반영한 정규분포에서 2만 번 시뮬레이션.
세 방법 차이가 크면 → 수익률 분포의 꼬리가 두껍다는 신호(정규분포 가정이 위험을 과소평가).

In [ ]:
nz = hd[stock_cols].gt(0).sum(axis=1)
rep = pd.Timestamp(REPORT_DATE) if REPORT_DATE else nz[nz >= 3].index[-1]
total = hd.loc[rep].sum()
w = (hd.loc[rep, stock_cols] / total)
w = w[w > 0].sort_values(ascending=False)
print(f"보고 기준일: {rep.date()} / 보유 {len(w)}종목 / 주식 비중 {w.sum():.0%} / 현금 {1 - w.sum():.0%}")
display(pd.DataFrame({"종목": [name(t) for t in w.index], "비중": w.values}, index=w.index)
        .style.format({"비중": "{:.1%}"}))

R = rk.position_returns(ind, list(w.index), rep, VAR_WINDOW)
pvar = rk.portfolio_var(R, w.values)
pvar.style.format("{:.2%}")

## ④ 리스크 기여도 (Component VaR)

- **VaR 기여 비율:** 전체 VaR 중 그 종목 몫 (합계 100%)
- **분산효과 = VaR 기여 비율 − 비중 비율:** 플러스면 비중보다 위험을 더 많이 가져오는 종목(집중 위험), 마이너스면 포트폴리오 위험을 낮춰주는 종목

In [ ]:
cv = rk.component_var(R, w.values)
cv.insert(0, "종목", [name(t) for t in cv.index])
display(cv.style.format({"비중": "{:.1%}", "단독 변동성(연)": "{:.1%}", "VaR 기여": "{:.2%}",
                         "VaR 기여 비율": "{:.1%}", "분산효과": "{:+.1%}"}))
ax = cv.set_index("종목")[["비중", "VaR 기여 비율"]].plot.barh(figsize=(8, 4), title="weight vs VaR contribution")
ax.invert_yaxis(); plt.show()
hhi = (w / w.sum()).pow(2).sum()
print(f"집중도(HHI): {hhi:.3f}  → 유효 종목 수 {1 / hhi:.1f}개")

## ⑤ 스트레스 테스트

- **역사적 시나리오:** 지금 포트폴리오가 그 사건을 그대로 다시 겪는다면? (그때 상장 전 종목은 베타 × KOSPI 로 대체)
- **전략의 실제 성과:** 그 기간에 전략이 실제로 얼마나 버텼나 (현금 비중 조절 효과 포함)
- **가상 시나리오:** KOSPI 가 −5/−10/−20/−30% 빠지면, 현재 포트폴리오 베타 기준 예상 손실

In [ ]:
betas = rk.betas_to(ind, list(w.index), kospi, rep)
port_beta = sum(w[t] * betas[t] for t in w.index)
st = rk.stress_historical(ind, kospi, w.to_dict(), betas, SCENARIOS)
st["전략 실제 성과"] = [rk.window_return(eq, s, e) for s, e in SCENARIOS.values()]
display(st.style.format({c: "{:+.1%}" for c in ["KOSPI", "현재 포트폴리오", "KOSPI 대비", "전략 실제 성과"]}))
print(f"현재 포트폴리오 베타(주식 비중 반영): {port_beta:.2f}")
rk.stress_hypothetical(port_beta).style.format("{:+.1%}")

## ⑥ 유동성 리스크

운용금액 AUM(기본 100억) 기준으로, 하루 거래대금의 10%씩만 판다고 가정할 때 전부 청산에 걸리는 일수.
대형주 위주라 대부분 1일 미만이어야 정상 — 길게 나오는 종목은 급락 때 못 팔 위험.

In [ ]:
liq = rk.liquidity(ind, w.to_dict(), rep, AUM, PARTICIPATION)
liq.insert(0, "종목", [name(t) for t in liq.index])
liq.style.format({"보유금액(억)": "{:.1f}", "20일 평균 거래대금(억)": "{:,.0f}", "청산 소요일": "{:.2f}"})

## ⑦ 한도 대시보드

In [ ]:
cur_dd = (eq.loc[:rep] / eq.loc[:rep].cummax() - 1).iloc[-1]
lim = rk.limit_table([
    ("1일 VaR99 (HS, 현재 포트폴리오)", pvar.loc["VaR 99%", "역사적 시뮬레이션"], LIMITS["VAR99"], "이하"),
    ("현재 낙폭 (고점 대비)", cur_dd, LIMITS["MDD"], "이상"),
    ("최대 단일 종목 비중", w.max(), LIMITS["SINGLE"], "이하"),
    ("최대 청산 소요일", liq["청산 소요일"].max(), LIMITS["LIQ_DAYS"], "이하"),
])
lim.style.format({"현재": "{:.2%}", "한도": "{:.2%}", "한도 사용률": "{:.0%}"})

## 요약 — 리스크 보고서 한 단락 (자동 생성)

In [ ]:
worst = st["현재 포트폴리오"].astype(float).idxmin()
b = bt_all.loc["HS VaR99"]
summary = f"""[{rep.date()} 기준 {RISK_STRAT} 전략 리스크 요약]
- 보유 {len(w)}종목, 주식 비중 {w.sum():.0%}, 포트폴리오 베타 {port_beta:.2f}
- 1일 VaR99: 역사적 {pvar.iloc[1, 0]:.2%} / 정규 {pvar.iloc[1, 1]:.2%} / 몬테카를로 {pvar.iloc[1, 2]:.2%},  ES97.5(역사적) {pvar.iloc[2, 0]:.2%}
- VaR 사후검증(HS 99%): 예외 {int(b['예외 수'])}회 (예상 {b['예상 예외 수']:.1f}회), Kupiec p={b['p-value']:.3f} → {b['판정(5%)']}, 최근 250일 {b['바젤 신호등']}
- 최대 위험 기여 종목: {cv.iloc[0]['종목']} (비중 {cv.iloc[0]['비중']:.1%} → VaR 기여 {cv.iloc[0]['VaR 기여 비율']:.1%})
- 최악 스트레스 시나리오: {worst} 재현 시 {st.loc[worst, '현재 포트폴리오']:+.1%} (KOSPI {st.loc[worst, 'KOSPI']:+.1%})
- 한도 상태: {', '.join(f"{i} {s}" for i, s in lim['상태'].items())}"""
print(summary)
open(f"results/03_risk_summary_{RISK_STRAT}.txt", "w").write(summary)

## 해석 메모 → experiments.md

- 1부: 유니버스를 100개로 넓혔을 때 02(상위 30)와 결과가 비슷한가? → 전략이 특정 종목에 의존하지 않는지 확인
- ① HS VaR 와 Param VaR 차이가 큰 시기는 언제였나? (꼬리 위험이 커진 시기)
- ② VaR 모델이 Kupiec 검정을 통과하나? 실패한다면 어느 쪽(과소/과대평가)인가?
- ③ 세 방법 중 어느 것이 가장 보수적인가, 그 이유는?
- ④ 위험이 소수 종목에 쏠려 있나?
- ⑤ 전략 실제 성과 vs 현재 포트폴리오 스트레스 손실 차이 = **현금 비중 조절(추세추종)의 방어 효과**
- `RISK_STRAT` 을 바꿔 전략별 리스크 프로필 비교
